In [1]:
import os
import random
import pandas as pd
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)

BASE_DIR = r"D:/PEC/HSI_Project"
SENTENCE_INDEX = os.path.join(BASE_DIR, "processed", "sentence_dataset_index.csv")
TENSOR_DIR = os.path.join(BASE_DIR, "tensors", "sentences")
PAIR_OUT = os.path.join(BASE_DIR, "processed", "pair_index.csv")

print("Sentence index exists:", os.path.exists(SENTENCE_INDEX))
print("Tensor dir exists:", os.path.exists(TENSOR_DIR))


Sentence index exists: True
Tensor dir exists: True


In [2]:
df = pd.read_csv(SENTENCE_INDEX)

print("Total sentence samples:", len(df))
print("Columns:", df.columns.tolist())

df.head()


Total sentence samples: 2307
Columns: ['page', 'document', 'image', 'pen', 'sentence', 'num_bands', 'height', 'width', 'path']


,page,document,image,pen,sentence,num_bands,height,width,path
0,0. Pages 1 and 2,Document_1,Image_01,Pen_1,Sentence_1,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
1,0. Pages 1 and 2,Document_1,Image_01,Pen_1,Sentence_2,149,78,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
2,0. Pages 1 and 2,Document_1,Image_01,Pen_2,Sentence_1,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
3,0. Pages 1 and 2,Document_1,Image_01,Pen_2,Sentence_2,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...
4,0. Pages 1 and 2,Document_1,Image_01,Pen_3,Sentence_1,149,77,515,D:/PEC/HSI_Project\raw_data\Segregated Sentenc...


In [3]:
df = df.reset_index(drop=True)
df["tensor"] = df.index.map(lambda i: f"sent_{i:05d}.pt")

# Verify tensor files exist
missing = [t for t in df["tensor"] if not os.path.exists(os.path.join(TENSOR_DIR, t))]
print("Missing tensor files:", len(missing))


Missing tensor files: 0


In [4]:
def sample_pair(df, same_writer=None, same_pen=None):
    """
    Returns (tensor_A, tensor_B, same_writer, same_pen)
    """
    a = df.sample(1).iloc[0]

    candidates = df.copy()

    if same_writer is not None:
        if same_writer:
            candidates = candidates[candidates["document"] == a["document"]]
        else:
            candidates = candidates[candidates["document"] != a["document"]]

    if same_pen is not None:
        if same_pen:
            candidates = candidates[candidates["pen"] == a["pen"]]
        else:
            candidates = candidates[candidates["pen"] != a["pen"]]

    candidates = candidates[candidates["tensor"] != a["tensor"]]

    if len(candidates) == 0:
        return None

    b = candidates.sample(1).iloc[0]

    return {
        "tensor_A": a["tensor"],
        "tensor_B": b["tensor"],
        "same_writer": int(a["document"] == b["document"]),
        "same_pen": int(a["pen"] == b["pen"]),
    }


In [5]:
TOTAL_PAIRS = 10000
pairs = []

TARGET_PER_TYPE = TOTAL_PAIRS // 4

configs = [
    {"same_writer": True,  "same_pen": None},
    {"same_writer": False, "same_pen": None},
    {"same_writer": None,  "same_pen": True},
    {"same_writer": None,  "same_pen": False},
]

for cfg in configs:
    collected = 0
    attempts = 0

    while collected < TARGET_PER_TYPE and attempts < TARGET_PER_TYPE * 20:
        pair = sample_pair(df, **cfg)
        attempts += 1

        if pair is not None:
            pairs.append(pair)
            collected += 1

    print(f"Collected {collected} pairs for config {cfg}")


Collected 2500 pairs for config {'same_writer': True, 'same_pen': None}
Collected 2500 pairs for config {'same_writer': False, 'same_pen': None}
Collected 2500 pairs for config {'same_writer': None, 'same_pen': True}
Collected 2500 pairs for config {'same_writer': None, 'same_pen': False}


In [6]:
pair_df = pd.DataFrame(pairs)

print("Total pairs:", len(pair_df))
pair_df.head()


Total pairs: 10000


,tensor_A,tensor_B,same_writer,same_pen
0,sent_02021.pt,sent_01539.pt,1,0
1,sent_00224.pt,sent_00230.pt,1,0
2,sent_01303.pt,sent_00001.pt,1,0
3,sent_00864.pt,sent_00867.pt,1,0
4,sent_02089.pt,sent_00754.pt,1,0


In [7]:
print("\nWriter label distribution:")
print(pair_df["same_writer"].value_counts())

print("\nPen label distribution:")
print(pair_df["same_pen"].value_counts())

# Check tensor existence
missing_A = pair_df[~pair_df["tensor_A"].apply(lambda x: os.path.exists(os.path.join(TENSOR_DIR, x)))]
missing_B = pair_df[~pair_df["tensor_B"].apply(lambda x: os.path.exists(os.path.join(TENSOR_DIR, x)))]

print("Missing tensor_A:", len(missing_A))
print("Missing tensor_B:", len(missing_B))



Writer label distribution:
same_writer
0    7435
1    2565
Name: count, dtype: int64

Pen label distribution:
same_pen
0    7352
1    2648
Name: count, dtype: int64
Missing tensor_A: 0
Missing tensor_B: 0


In [8]:
pair_df.to_csv(PAIR_OUT, index=False)

print("✔ Pair index saved to:")
print(PAIR_OUT)


✔ Pair index saved to:
D:/PEC/HSI_Project\processed\pair_index.csv
